In [1]:
import os
import re
import json
import pandas as pd
import numpy as np
from itertools import combinations
import matplotlib.pyplot as plt

In [2]:
os.environ['HTTP_PROXY'] = 'http://proxy.is.depaul.edu:3128'
os.environ['HTTPS_PROXY'] = 'http://proxy.is.depaul.edu:3128'

In [3]:
from transformers import AutoModel, AutoTokenizer, AutoConfig, BertTokenizerFast
import torch.nn.functional as F
import torch

In [4]:
model_name = 'allenai/scibert_scivocab_uncased'
model = AutoModel.from_pretrained(model_name, output_hidden_states=True)
tok = AutoTokenizer.from_pretrained(model_name)

'timed out' thrown while requesting HEAD https://huggingface.co/allenai/scibert_scivocab_uncased/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'timed out' thrown while requesting HEAD https://huggingface.co/allenai/scibert_scivocab_uncased/resolve/main/config.json
Retrying in 2s [Retry 2/5].
'timed out' thrown while requesting HEAD https://huggingface.co/allenai/scibert_scivocab_uncased/resolve/main/config.json
Retrying in 4s [Retry 3/5].
'timed out' thrown while requesting HEAD https://huggingface.co/allenai/scibert_scivocab_uncased/resolve/main/config.json
Retrying in 8s [Retry 4/5].


KeyboardInterrupt: 

In [4]:
model = model.to('cuda')

In [5]:
sentence = 'There is something wierd going on here.'

In [6]:
texts = [sentence]

In [46]:
def tokenize_and_find(texts, keywords):
    if isinstance(keywords, str):
        keywords = [keywords]

    keyword_id_groups = []
    for kw in keywords:
        tokens = tok.tokenize(kw)
        ids = tok.convert_tokens_to_ids(tokens)
        keyword_id_groups.append(ids)

    inputs = tok(texts,
                max_length=512,
                return_offsets_mapping=True,
                add_special_tokens=False,
                return_tensors="pt",
                padding=True,
                truncation=True)

    results = []
    for i in range(len(texts)):
        token_ids = inputs['input_ids'][i].tolist()
        match_positions = []

        for keyword_ids in keyword_id_groups:
            for j in range(len(token_ids) - len(keyword_ids) + 1):
                if token_ids[j:j + len(keyword_ids)] == keyword_ids:
                    match_positions.append(tuple(range(j, j + len(keyword_ids))))

        if match_positions:
            results.append([i, match_positions])

    return results

In [48]:
def extract_target_embeddings(embs, found_indices):
    
    selected_embs = []
    for i in range(len(found_indices)):
        for tok_idx in found_indices[i][1]:
            selected_embs += [embs[i,list(tok_idx),:].mean(dim=0)]
            
    return selected_embs

In [87]:
texts = ['covid was a disease', 'I had included covid in my research in a covid-related paper']
keyword = 'covid'
I,A,H0s = simply_embed(texts)
H0s = H0s['hidden_states'][0]
#
keyword_matches = tokenize_and_find(texts, keyword)
first_H0 = extract_target_embeddings(H0s, keyword_matches)

In [111]:
keyword_layers_embed(texts, keyword, layers=0,model=model)

tensor([[-0.0291, -0.0291, -0.0291],
        [-0.0237, -0.0237, -0.0237],
        [-0.0247, -0.0247, -0.0247],
        ...,
        [ 0.0035,  0.0035,  0.0035],
        [-0.0328, -0.0328, -0.0328],
        [ 0.0151,  0.0151,  0.0151]], grad_fn=<StackBackward0>)

In [19]:
# seg-id embedding
seg_embs = model.embeddings.token_type_embeddings(torch.zeros_like(I))
# position embedding
position_ids = torch.arange(I.shape[1]).unsqueeze(0).expand(1, -1).to('cuda')
position_embeddings = model.embeddings.position_embeddings(position_ids)

In [69]:
# for having a truly decontextualized embeddings we'll need to use
# model.embeddings.word_embeddings
model.embeddings.word_embeddings(I)[0,2,:]

tensor([-7.2439e-03,  4.8421e-02, -4.4720e-02, -1.3058e-02, -2.6661e-03,
        -5.1116e-02,  7.3844e-03, -3.4637e-02, -6.4904e-02,  2.2736e-02,
        -4.0520e-02, -1.3779e-02,  1.8802e-02,  1.7801e-02, -1.9137e-02,
        -8.6514e-03, -3.6196e-03, -4.9315e-02, -8.4805e-03, -6.2794e-03,
         2.6174e-02, -5.8018e-02,  2.5185e-03, -1.7491e-02, -2.9540e-02,
         3.5104e-03, -4.0021e-02,  4.0856e-02, -8.9123e-02,  4.1832e-02,
        -1.6988e-02, -2.7266e-02,  5.7238e-02,  1.7584e-04, -3.3000e-02,
         4.7781e-02,  4.3964e-03, -1.0093e-02,  4.5166e-02, -1.3353e-02,
        -6.6938e-02, -4.2525e-03,  2.9657e-02,  1.3551e-02, -4.7799e-02,
        -3.0138e-03, -2.0097e-03,  8.5363e-02, -5.9306e-02,  2.6010e-02,
        -1.9785e-02, -1.5311e-02,  2.9259e-02,  2.8888e-02, -8.9563e-03,
         1.8168e-02,  4.0439e-02,  1.3603e-03, -2.4250e-02, -4.8589e-02,
         3.0441e-03,  4.6496e-02,  7.4185e-03, -2.2372e-02,  3.9587e-02,
        -7.2164e-02, -3.7071e-02, -5.9216e-02, -1.3

# Experimenting with BATS

In [7]:
PATH_BATS_gender_file = "/home/jsourati/embeddings/data/BATS_3.0/3_Encyclopedic_semantics/E10 [male - female].txt"

In [8]:
e10_df = pd.read_csv(PATH_BATS_gender_file, sep='\t', header=None)

In [9]:
e10_df[1] = e10_df[1].apply(lambda x: x.split('/')[0])

In [10]:
pool = e10_df[0].tolist() + e10_df[1].tolist()

In [154]:
pool = np.array(pool)

In [561]:
encoding = tok.batch_encode_plus(
            ['neice', 'grandson', 'granddaughter'],
            max_length=512,
            return_offsets_mapping=True,
            add_special_tokens=False,
            return_tensors="pt",
            padding=True,
            truncation=True)

inputs = {k: encoding[k].to('cuda') for k in ["input_ids", "token_type_ids", "attention_mask"] if k in encoding}
input_ids = inputs["input_ids"]

In [562]:
for i in range(input_ids.shape[0]):
    for j in range(input_ids.shape[1]):
        print(tok.decode(input_ids[i,j]), end=' ')
    print()

ne ##ice [PAD] 
grand ##son [PAD] 
grand ##da ##ughter 


In [419]:
def simply_embed(texts):

    encoding = tok.batch_encode_plus(
                list(texts),
                max_length=512,
                return_offsets_mapping=True,
                add_special_tokens=False,
                return_tensors="pt",
                padding=True,
                truncation=True)
    
    inputs = {k: encoding[k].to('cuda') for k in ["input_ids", "token_type_ids", "attention_mask"] if k in encoding}
    input_ids = inputs["input_ids"]
    # the unsqueezing is to enable us to multiply it with hidden outputs
    attention_mask = inputs["attention_mask"].unsqueeze(-1).to('cpu')
    
    with torch.no_grad():
        outputs = model(**inputs)
        
    return input_ids, attention_mask, outputs


def layers_embed(texts, layers=[-1], model=None):
    '''the model is needed only if `layers` is set to 0 (outside a list)
    
    layers=[0] means we are using "contextualized" input embedding (contextualized because it has
    the positional and segment-ID embeddings, making it slightly contextualized)
    layers=0 means truly decontextualized embeddings
    '''
    
    tok_IDs,attention_mask,outputs = simply_embed(texts)
        
    # taking the layer-wise average first
    if layers==0:
        with torch.no_grad():
            hidden_outputs = model.embeddings.word_embeddings(tok_IDs).to('cpu')
    else:
        hidden_outputs = torch.stack(outputs['hidden_states'], dim=-1).to('cpu')
        hidden_outputs = hidden_outputs[:,:,:,layers].mean(axis=-1)
    
    # now, taking token-wise average excluding the masked tokens
    hidden_outputs = hidden_outputs*attention_mask
    hidden_outputs = hidden_outputs.sum(dim=1) / attention_mask.sum(dim=1)
    
    return hidden_outputs


def keyword_layers_embed(texts, keyword, layers=[-1], model=None):
    '''the model is needed only if `layers` is set to 0 (outside a list)
    
    layers=[0] means we are using "contextualized" input embedding (contextualized because it has
    the positional and segment-ID embeddings, making it slightly contextualized)
    layers=0 means truly decontextualized embeddings
    '''
    
    tok_IDs,attention_mask,outputs = simply_embed(texts)
        
    # taking the layer-wise average first
    if layers==0:
        with torch.no_grad():
            hidden_outputs = model.embeddings.word_embeddings(tok_IDs).to('cpu')
    else:
        hidden_outputs = torch.stack(outputs['hidden_states'], dim=-1).to('cpu')
        hidden_outputs = hidden_outputs[:,:,:,layers].mean(axis=-1)
        
    # finding out which parts of the outputs we are interested in
    keyword_matches = tokenize_and_find(texts, keyword)
    kw_hidden_outputs = extract_target_embeddings(hidden_outputs, keyword_matches)
    
    return torch.stack(kw_hidden_outputs,dim=0)
    

In [173]:
H = torch.stack(outputs['hidden_states'], dim=-1)[:,:,:,layers].mean(axis=-1).to('cpu')
A = attention_mask.unsqueeze(-1).to('cpu')
Hp = H*A
Hpp = Hp.sum(dim=1) / A.sum(dim=1)
Hp.sum(axis=1)[1,:] / Hpp[1,:]

In [13]:
def get_similarities(row1,row2,embs):
    
    pair_11, pair_12 = e10_df.iloc[row1,:].tolist()
    pair_21, pair_22 = e10_df.iloc[row2,:].tolist()
    
    E11 = embs[pool==pair_11,:]
    E12 = embs[pool==pair_12,:]
    E21 = embs[pool==pair_21,:]
    E22 = embs[pool==pair_22,:]
    
    # query embedding
    Eq = E21 - E22 + E12
    
    Eq = F.normalize(Eq, dim=-1)
    embs  = F.normalize(embs, dim=-1)
    cosims = (Eq @ embs.T).squeeze()
    
    return cosims, (pair_11,pair_12,pair_21,pair_22)

In [14]:
def get_target_rank(sims, target):
    '''the `sims` should be of the same order as the `pool` list'''
    
    sorted_pool = pool[torch.argsort(sims, descending=True)]
    
    if target not in sorted_pool:
        return np.nan
    else:
        return np.where(sorted_pool==target)[0][0]

In [149]:
# pairwise combinations of indices
combs = list(combinations(range(len(e10_df)), 2))
np.random.shuffle(combs)

In [181]:
layers_dict = {'first':[0], 'first_two':[0,1],  'first_three':[0,1,2],
               'last': [-1], 'last_two':[-2,-1], 'last_three':[-3,-2,-1]}
embs_dict = {}
ranks_dict = {}
for name,layers in layers_dict.items():
    print(f'Working on {name}...')
    # treat layer 0 differently
    if 0 in layers:
        E = layers_embed(pool, layers=0, model=model).unsqueeze(dim=-1)
        layers.remove(0)
        # if layers other than 0, just concatenate
        for l in layers:
            more_E = layers_embed(pool, layers=[l]).unsqueeze(dim=-1)
            E = torch.concatenate((E, more_E), dim=-1)
        E = E.mean(dim=-1)
            
    else:
        E = layers_embed(pool, layers=layers, model=model)
        
    embs_dict[name] = E
            
    ranks = []
    for t in range(1000):
        i,j = combs[t]
        sims,pairs = get_similarities(i,j,embs_dict[name])
        ranks += [get_target_rank(sims, pairs[0])]
    ranks_dict[name] = np.array(ranks)

Working on first...
Working on first_two...
Working on first_three...
Working on last...
Working on last_two...
Working on last_three...


In [194]:
def top_k_accuracy(ranks, k=1):
    return sum(ranks<=k)/len(ranks)

In [208]:
for name, ranks in ranks_dict.items():
    top1 = top_k_accuracy(ranks, k=1)
    top2 = top_k_accuracy(ranks, k=2)
    top5 = top_k_accuracy(ranks, k=5)
    
    print('{:10}\t{:5.2f}\t{:5.2f}\t{:5.2f}'.format(name, top1, top2, top5))

first     	 0.17	 0.40	 0.61
first_two 	 0.16	 0.37	 0.57
first_three	 0.23	 0.43	 0.62
last      	 0.19	 0.40	 0.54
last_two  	 0.18	 0.37	 0.54
last_three	 0.18	 0.37	 0.54


In [ ]:
p

In [ ]:
def process_tree(self, tree_dict:dict):
    num_nodes = 0

    embedding_coeff = {}    # the format for data key is - [node's depth, node's num of occurances, the response from LLM]
    def process_nodes(tree_dict:dict):
        nonlocal num_nodes
        for keyword, node_dict in tree_dict.items():
            if node_dict.get("response"):
                num_nodes += 1
            # Using formula for weight = [(1 * # of occurances) / (depth of the node)] * embedding:
            weight_coeff = (1 * embedding_coeff[keyword]["data"][1])/(embedding_coeff[keyword]["data"][0])

            # Now recurse-
            if (len(node_dict['children']) != 0):
                process_nodes(node_dict["children"])
    process_nodes(tree_dict=tree_dict)
    return embedding_coeff, num_nodes

In [370]:
with open("./data/BATS/male_female/man/run_1/tree.json", "r") as f:
    jsonT = json.load(f)

In [382]:
def prune_context(context):
    pruned_context = re.sub('definition:+', '', context.lower())
    return pruned_context.strip()

def process_nodes(nodes, parent="NA"):
    nodes_dict = {}
    for name,nd in nodes.items():
        nodes_dict[name] = {"context": prune_context(nd['response']), 'parent': parent}
        
    return nodes_dict

In [383]:
def get_tree_dict(keyword_json, max_level=None):

    tree_dict = {}

    # (parent, child_node): the root node does not have any parent
    children = [(keyword_json, 'NA')]
    cnt = 1
    tree_dict = {}
    while len(children)>0:
        if max_level:
            if cnt>max_level:
                break
            
        tree_dict[cnt] = {}
        new_children = []
        for child in children:
            processed_dict = process_nodes(child[0], child[1])
            tree_dict[cnt].update(processed_dict) # adding the result of processing this node

            # updating the new children nodes
            for name,branches in child[0].items():
                if len(branches['children'])>0:
                    new_children += [(branches['children'], name)]

        children = new_children.copy()
        cnt += 1
        
    return tree_dict

In [426]:
T = get_tree_dict(jsonT, max_level=4)

In [390]:
texts = [T[1]['man']['context']]

In [430]:
# apparently the tree is not really a tree as it has loop (repeating keywords in different levels)
set(T[4]).intersection(set(T[2]))

{'decision-making', 'leadership'}

In [336]:
fnames = [x for x in os.listdir("./data/BATS/male_female")]

In [391]:
encoding = tok.batch_encode_plus(
            texts,
            max_length=512,
            return_offsets_mapping=True,
            add_special_tokens=False,
            return_tensors="pt",
            padding=True,
            truncation=True)

In [478]:
def extract_embedding_from_tree(tree, max_level=None, layers=[-1], model=None, verbose=True):
    '''As before, the model is needed only if `layers` is set to 0 (outside a list)'''
    
    kw = list(tree[1])[0]
    
    total_E = 0
    total_w = 0
    
    for level, vals in tree.items():
        if max_level:
            if level>max_level:
                break
                
        for name,content in vals.items(): 
            E = keyword_layers_embed([content['context']], name, layers=layers, model=model)
            # updating the weighted sum of E's and sum of the weights
            total_E += E/level
            total_w += 1/level
            
            # just to check
            if verbose:
                print("Level {}, {:20}, current total weight={:.2f}".format(level, name, total_w))
            
    return total_E/total_w

def extract_multiple_embeddings_from_trees(trees, max_level=None, layers=[-1], model=None):
    
    E = []
    for T in trees:
        E += [extract_embedding_from_tree(T, 
                                          max_level=max_level, 
                                          layers=layers, 
                                          model=model, 
                                          verbose=False)]
        
    return torch.concatenate(E, dim=0)

In [483]:
extract_embedding_from_tree(pool_trees_lst[0], 
                          max_level=3, 
                          layers=0, 
                          model=model, 
                          verbose=False)

RuntimeError: stack expects a non-empty TensorList

In [489]:
for name,content in pool_trees_lst[0][2].items(): 
    print(name)
    keyword_layers_embed([content['context']], name, layers=layers, model=model)

entity
performs


RuntimeError: stack expects a non-empty TensorList

In [514]:
bad_cases = []
for i,tree in enumerate(pool_trees_lst):
    for cnt in tree:
        bad_cases += [(pool[i], cnt, x) for x,y in tree[cnt].items() if x.lower() not in y['context']]

In [520]:
len(bad_cases)

26

In [521]:
bad_cases[:10]

[('actor', 2, 'performs'),
 ('actor', 3, 'performances'),
 ('batman', 3, 'gather evidence'),
 ('boy', 1, 'boy'),
 ('brother', 1, 'brother'),
 ('brother', 3, 'single person'),
 ('buck', 3, 'stores energy'),
 ('businessman', 3, 'innovates'),
 ('chairman', 3, 'topics'),
 ('chairman', 3, 'roles')]

In [522]:
pool_trees['boy'][1]['boy']

{'context': 'a male child, typically under the age of puberty, characterized by youthfulness and developmental growth phases.',
 'parent': 'NA'}

In [450]:
maxl = 3

pool_trees = {}
for kw in pool:
    with open(f"./data/BATS/male_female/{kw}/run_1/tree.json", "r") as f:
        jsonT = json.load(f)
    pool_trees[kw] = get_tree_dict(jsonT, max_level=maxl)

In [462]:
pool_trees_lst = [pool_trees[x] for x in pool]

In [479]:
# root only

layers_dict = {'first':[0], 'first_two':[0,1],  'first_three':[0,1,2],
               'last': [-1], 'last_two':[-2,-1], 'last_three':[-3,-2,-1]}
embs_dict = {}
ranks_dict = {}
for name,layers in layers_dict.items():
    print(f'Working on {name}...')
    # treat layer 0 differently
    if 0 in layers:
        E = extract_multiple_embeddings_from_trees(pool_trees_lst, 
                                                   max_level=maxl,
                                                   layers=0, 
                                                   model=model).unsqueeze(dim=-1)
        layers.remove(0)
        # if layers other than 0, just concatenate
        for l in layers:
            more_E = extract_multiple_embeddings_from_trees(pool_trees_lst, 
                                                            max_level=maxl,
                                                            layers=[l]).unsqueeze(dim=-1)
            E = torch.concatenate((E, more_E), dim=-1)
        E = E.mean(dim=-1)
            
    else:
        E = extract_multiple_embeddings_from_trees(pool_trees_lst, 
                                                   max_level=maxl,
                                                   layers=layers, 
                                                   model=model)
        
    embs_dict[name] = E
            
    ranks = []
    for t in range(1000):
        i,j = combs[t]
        sims,pairs = get_similarities(i,j,embs_dict[name])
        ranks += [get_target_rank(sims, pairs[0])]
    ranks_dict[name] = np.array(ranks)

Working on first...


RuntimeError: stack expects a non-empty TensorList

In [465]:
for name, ranks in ranks_dict.items():
    top1 = top_k_accuracy(ranks, k=1)
    top2 = top_k_accuracy(ranks, k=2)
    top5 = top_k_accuracy(ranks, k=5)
    
    print('{:10}\t{:5.2f}\t{:5.2f}\t{:5.2f}'.format(name, top1, top2, top5))

first     	 0.00	 0.00	 0.00
first_two 	 0.00	 0.00	 0.00
first_three	 0.00	 0.00	 0.00
last      	 0.00	 0.00	 0.00
last_two  	 0.00	 0.00	 0.00
last_three	 0.00	 0.00	 0.00


In [468]:
embs_dict['first'].shape

torch.Size([100, 768])